In [1]:
import pandas as pd
import os

print("pandas version:", pd.__version__)
print("Ready to go!")

pandas version: 3.0.3
Ready to go!


In [2]:
files = os.listdir('../data/')
print(files)

['archive.zip', 'Budget.csv', 'income_clean.csv', 'Personal_transactions.csv', 'spending_clean.csv']


In [6]:

# Load both files
budget = pd.read_csv('../data/Budget.csv')
transactions = pd.read_csv('../data/Personal_transactions.csv')

print("\n=== BUDGET ===")
print("Shape:", budget.shape)
print("Columns:", budget.columns.tolist())
print(budget.head(3))

print("\n=== TRANSACTIONS ===")
print("Shape:", transactions.shape)
print("Columns:", transactions.columns.tolist())
print(transactions.head(5))
      


=== BUDGET ===
Shape: (19, 2)
Columns: ['Category', 'Budget']
         Category  Budget
0  Alcohol & Bars      50
1  Auto Insurance      75
2    Coffee Shops      15

=== TRANSACTIONS ===
Shape: (806, 6)
Columns: ['Date', 'Description', 'Amount', 'Transaction Type', 'Category', 'Account Name']
         Date          Description   Amount Transaction Type  \
0  01/01/2018               Amazon    11.11            debit   
1  01/02/2018     Mortgage Payment  1247.44            debit   
2  01/02/2018      Thai Restaurant    24.22            debit   
3  01/03/2018  Credit Card Payment  2298.09           credit   
4  01/04/2018              Netflix    11.76            debit   

              Category   Account Name  
0             Shopping  Platinum Card  
1      Mortgage & Rent       Checking  
2          Restaurants    Silver Card  
3  Credit Card Payment  Platinum Card  
4        Movies & DVDs  Platinum Card  


In [7]:
# Let's understand our data deeper
print("=== DATE RANGE ===")
print(transactions['Date'].min(), "to", transactions['Date'].max())

print("\n=== TRANSACTION TYPES ===")
print(transactions['Transaction Type'].value_counts())

print("\n=== CATEGORIES ===")
print(transactions['Category'].value_counts())

print("\n=== ACCOUNTS ===")
print(transactions['Account Name'].value_counts())

print("\n=== ANY MISSING VALUES? ===")
print(transactions.isnull().sum())

=== DATE RANGE ===
01/01/2018 to 12/29/2018

=== TRANSACTION TYPES ===
Transaction Type
debit     688
credit    118
Name: count, dtype: int64

=== CATEGORIES ===
Category
Credit Card Payment       143
Groceries                 105
Restaurants                81
Utilities                  63
Shopping                   60
Gas & Fuel                 52
Paycheck                   46
Home Improvement           36
Coffee Shops               31
Alcohol & Bars             25
Mortgage & Rent            21
Music                      21
Mobile Phone               21
Internet                   21
Movies & DVDs              18
Auto Insurance             18
Fast Food                  16
Haircut                    13
Television                  8
Electronics & Software      4
Food & Dining               2
Entertainment               1
Name: count, dtype: int64

=== ACCOUNTS ===
Account Name
Platinum Card    366
Checking         264
Silver Card      176
Name: count, dtype: int64

=== ANY MISSING VALUES

In [8]:
# ============================================
# DATA CLEANING PIPELINE
# ============================================

# Step 1 — Convert Date from string to datetime
transactions['Date'] = pd.to_datetime(transactions['Date'])

print("✅ Step 1 — Date converted")
print(transactions['Date'].dtype)

# Step 2 — Extract useful time columns
transactions['Month'] = transactions['Date'].dt.month
transactions['Month_Name'] = transactions['Date'].dt.strftime('%B')
transactions['Year'] = transactions['Date'].dt.year

print("\n✅ Step 2 — Month and Year columns added")
print(transactions[['Date', 'Month', 'Month_Name', 'Year']].head(3))

# Step 3 — Separate spending from non-spending
non_spending = ['Credit Card Payment', 'Paycheck']

spending_df = transactions[~transactions['Category'].isin(non_spending)].copy()
income_df = transactions[transactions['Category'] == 'Paycheck'].copy()

print(f"\n✅ Step 3 — Removed non-spending rows")
print(f"Original rows: {len(transactions)}")
print(f"Spending rows: {len(spending_df)}")
print(f"Income rows:   {len(income_df)}")

# Step 4 — Keep only debits for spending analysis
spending_df = spending_df[spending_df['Transaction Type'] == 'debit'].copy()

print(f"\n✅ Step 4 — Kept only debit transactions")
print(f"Final spending rows: {len(spending_df)}")

# Step 5 — Fix small categories
spending_df['Category'] = spending_df['Category'].replace({
    'Food & Dining': 'Restaurants',
    'Entertainment': 'Movies & DVDs'
})

print("\n✅ Step 5 — Small categories merged")
print(spending_df['Category'].value_counts())


✅ Step 1 — Date converted
datetime64[us]

✅ Step 2 — Month and Year columns added
        Date  Month Month_Name  Year
0 2018-01-01      1    January  2018
1 2018-01-02      1    January  2018
2 2018-01-02      1    January  2018

✅ Step 3 — Removed non-spending rows
Original rows: 806
Spending rows: 617
Income rows:   46

✅ Step 4 — Kept only debit transactions
Final spending rows: 617

✅ Step 5 — Small categories merged
Category
Groceries                 105
Restaurants                83
Utilities                  63
Shopping                   60
Gas & Fuel                 52
Home Improvement           36
Coffee Shops               31
Alcohol & Bars             25
Mortgage & Rent            21
Music                      21
Mobile Phone               21
Internet                   21
Movies & DVDs              19
Auto Insurance             18
Fast Food                  16
Haircut                    13
Television                  8
Electronics & Software      4
Name: count, dtype: int64

In [9]:
# ============================================
# SUMMARY STATISTICS
# ============================================

print("=== OVERALL SPENDING SUMMARY ===")
print(f"Total spent in 2018:        ${spending_df['Amount'].sum():,.2f}")
print(f"Average transaction:        ${spending_df['Amount'].mean():,.2f}")
print(f"Largest transaction:        ${spending_df['Amount'].max():,.2f}")
print(f"Smallest transaction:       ${spending_df['Amount'].min():,.2f}")
print(f"Total transactions:         {len(spending_df)}")

print("\n=== MONTHLY SPENDING SUMMARY ===")
monthly_sorted = spending_df.groupby(['Month', 'Month_Name'])['Amount'].sum().reset_index()
monthly_sorted = monthly_sorted.sort_values('Month')
monthly_sorted['Amount'] = monthly_sorted['Amount'].round(2)
print(monthly_sorted[['Month_Name', 'Amount']].to_string(index=False))

print("\n=== TOP 5 SPENDING CATEGORIES ===")
top_cats = spending_df.groupby('Category')['Amount'].sum().sort_values(ascending=False).head(5)
for cat, amt in top_cats.items():
    print(f"  {cat:<25} ${amt:,.2f}")

print("\n=== TOP 5 MERCHANTS ===")
top_merchants = spending_df.groupby('Description')['Amount'].sum().sort_values(ascending=False).head(5)
for merchant, amt in top_merchants.items():
    print(f"  {merchant:<25} ${amt:,.2f}")


=== OVERALL SPENDING SUMMARY ===
Total spent in 2018:        $63,042.42
Average transaction:        $102.18
Largest transaction:        $9,200.00
Smallest transaction:       $1.75
Total transactions:         617

=== MONTHLY SPENDING SUMMARY ===
Month_Name   Amount
   January  3803.08
  February  4044.04
     March  4321.25
     April  4829.53
       May 12760.97
      June 13027.43
      July  4853.76
    August  4035.88
 September  4353.02
   October  2242.37
  November  2118.15
  December  2652.94

=== TOP 5 SPENDING CATEGORIES ===
  Mortgage & Rent           $24,754.50
  Home Improvement          $19,092.87
  Groceries                 $2,795.21
  Utilities                 $2,776.00
  Restaurants               $2,690.77

=== TOP 5 MERCHANTS ===
  Mortgage Payment          $24,754.50
  Mike's Construction Co.   $17,200.00
  Grocery Store             $2,764.33
  Amazon                    $1,970.04
  Hardware Store            $1,892.87


In [10]:
# ============================================
# SAVE CLEAN DATA
# ============================================

# Save cleaned spending data
spending_df.to_csv('../data/spending_clean.csv', index=False)

# Save income separately
income_df.to_csv('../data/income_clean.csv', index=False)

print("✅ Clean data saved successfully")
print(f"   spending_clean.csv — {len(spending_df)} rows")
print(f"   income_clean.csv  — {len(income_df)} rows")


✅ Clean data saved successfully
   spending_clean.csv — 617 rows
   income_clean.csv  — 46 rows
